In [1]:
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np
from config import NUMERIC_PREDICTORS, CATEGORICAL_PREDICTORS,EXPERIMENTS,RESULTS_DIR

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# ADNI Target Feature Schema
NUMERIC_PREDICTORS = [
    "AGE", "PTEDUCAT", "APOE4",
    "ADAS11_bl", "ADAS13_bl", "ADASQ4_bl",
    "RAVLT_immediate_bl", "RAVLT_learning_bl", "RAVLT_forgetting_bl", "RAVLT_perc_forgetting_bl",
    "DIGITSCOR_bl", "TRABSCOR_bl", "MOCA_bl"
]

CATEGORICAL_PREDICTORS = ["Sex"]
TARGET_COL = "Group"

def build_oasis3_harmonized_clinical(
    metadata_dir: str = "/home/ybrima/Data/oasis3_metadata",
    output_csv: str = "oasis3_harmonized_clinical.csv"
):
    meta_path = Path(metadata_dir)
    
    print("1. Loading raw OASIS-3 tables...")
    df_cdr = pd.read_csv(meta_path / 'OASIS3_UDSb4_cdr.csv')
    df_demo = pd.read_csv(meta_path / 'OASIS3_demographics.csv')
    
    cog_path = meta_path / 'OASIS3_UDSc1_cognitive_assessments.csv'
    df_cog = pd.read_csv(cog_path) if cog_path.exists() else pd.DataFrame()

    records = []

    print("2. Mapping fields to ADNI schema...")
    for idx, cdr_row in df_cdr.iterrows():
        subj = cdr_row['OASISID']
        cdr_days = cdr_row['days_to_visit']
        
        # --- Target Label: Diagnosis from CDRTOT ---
        cdr_tot = cdr_row.get('CDRTOT')
        if pd.isna(cdr_tot):
            continue
            
        cdr_tot = float(cdr_tot)
        if cdr_tot == 0.0:
            diag = "CN"
        elif cdr_tot == 0.5:
            diag = "MCI"
        elif cdr_tot >= 1.0:
            diag = "AD"
        else:
            continue

        # --- Demographics & Genetics ---
        subj_demo = df_demo[df_demo['OASISID'] == subj]
        if subj_demo.empty:
            continue
        demo_row = subj_demo.iloc[0]
        
        # Sex (1 -> Male, 2 -> Female)
        raw_gender = demo_row.get('GENDER')
        if raw_gender == 1 or str(raw_gender).strip() == '1':
            sex = 'Male'
        elif raw_gender == 2 or str(raw_gender).strip() == '2':
            sex = 'Female'
        else:
            sex = np.nan
            
        # Age at Visit
        age = cdr_row.get('age at visit', np.nan)
        
        # Education
        education = demo_row.get('EDUC', np.nan)
        
        # APOE4 Allele Count (e.g., 23 -> 0, 34 -> 1, 44 -> 2)
        apoe_val = str(demo_row.get('APOE', ''))
        apoe4_count = apoe_val.count('4') if apoe_val and apoe_val != 'nan' else np.nan

        # --- Cognitive Tests (Match nearest visit <= 365 days) ---
        moca = np.nan
        trabscor = np.nan
        digitscor = np.nan
        
        if not df_cog.empty:
            subj_cog = df_cog[df_cog['OASISID'] == subj].copy()
            if not subj_cog.empty:
                subj_cog['diff'] = (subj_cog['days_to_visit'] - cdr_days).abs()
                closest_cog = subj_cog.loc[subj_cog['diff'].idxmin()]
                
                if closest_cog['diff'] <= 365:
                    moca = closest_cog.get('mocatots', np.nan)
                    trabscor = closest_cog.get('tmb', closest_cog.get('trailb', np.nan))
                    digitscor = closest_cog.get('digsym', np.nan)

        # --- Build Harmonized ADNI Record ---
        record = {
            "subject_id": subj,
            "Visit_Day": cdr_days,
            "OASIS_Session": cdr_row.get('OASIS_session_label'),
            "Group": diag,
            "Sex": sex,
            "AGE": age,
            "PTEDUCAT": education,
            "APOE4": apoe4_count,
            "MOCA_bl": moca,
            "TRABSCOR_bl": trabscor,
            "DIGITSCOR_bl": digitscor,
            # ADNI-specific tests not in UDS populated as NaN
            "ADAS11_bl": np.nan,
            "ADAS13_bl": np.nan,
            "ADASQ4_bl": np.nan,
            "RAVLT_immediate_bl": np.nan,
            "RAVLT_learning_bl": np.nan,
            "RAVLT_forgetting_bl": np.nan,
            "RAVLT_perc_forgetting_bl": np.nan,
        }
        records.append(record)

    df_out = pd.DataFrame(records)
    
    # Ensure standard column order
    all_cols = ["subject_id", "Visit_Day", "OASIS_Session", TARGET_COL] + CATEGORICAL_PREDICTORS + NUMERIC_PREDICTORS
    df_out = df_out.reindex(columns=all_cols)
    
    df_out.to_csv(RESULTS_DIR / output_csv, index=False)
    
    print("\n--- Harmonization Complete ---")
    print(f"Total Harmonized Records Saved: {len(df_out)}")
    print("\nClass Distribution:")
    print(df_out["Group"].value_counts())
    print("\nFeature Missingness Summary:")
    print(df_out[NUMERIC_PREDICTORS + CATEGORICAL_PREDICTORS].isnull().sum())
    
    return df_out


df_out = build_oasis3_harmonized_clinical()

1. Loading raw OASIS-3 tables...
2. Mapping fields to ADNI schema...



--- Harmonization Complete ---
Total Harmonized Records Saved: 8625

Class Distribution:
Group
CN     6479
MCI    1444
AD      702
Name: count, dtype: int64

Feature Missingness Summary:
AGE                            0
PTEDUCAT                       8
APOE4                         59
ADAS11_bl                   8625
ADAS13_bl                   8625
ADASQ4_bl                   8625
RAVLT_immediate_bl          8625
RAVLT_learning_bl           8625
RAVLT_forgetting_bl         8625
RAVLT_perc_forgetting_bl    8625
DIGITSCOR_bl                2305
TRABSCOR_bl                 1460
MOCA_bl                     5975
Sex                            0
dtype: int64


In [9]:
df_out.head()

,subject_id,Visit_Day,OASIS_Session,Group,Sex,AGE,PTEDUCAT,APOE4,ADAS11_bl,ADAS13_bl,ADASQ4_bl,RAVLT_immediate_bl,RAVLT_learning_bl,RAVLT_forgetting_bl,RAVLT_perc_forgetting_bl,DIGITSCOR_bl,TRABSCOR_bl,MOCA_bl
0,OAS30001,0,OAS30001_UDSb4_d0000,CN,Female,65.19,12.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.0,64.0,NaN
1,OAS30001,339,OAS30001_UDSb4_d0339,CN,Female,66.12,12.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.0,64.0,NaN
2,OAS30001,722,OAS30001_UDSb4_d0722,CN,Female,67.17,12.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.0,73.0,NaN
3,OAS30001,1106,OAS30001_UDSb4_d1106,CN,Female,68.22,12.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,74.0,NaN
4,OAS30001,1456,OAS30001_UDSb4_d1456,CN,Female,69.18,12.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.0,82.0,NaN


In [20]:
import os
import re
from pathlib import Path
import pandas as pd

# 1. Define paths
NIFTI_DIR = Path("/home/ybrima/Data/OASIS3")
RESULTS_DIR = Path("./results")  # Adjust if you have a specific results directory
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Collect all NIfTI files
all_nifti_files = [
    Path(root) / f
    for root, _, files in os.walk(NIFTI_DIR)
    for f in files
    if f.endswith(".nii") or f.endswith(".nii.gz")
]

print(f"Total NIfTI files found: {len(all_nifti_files)}")

data = []

# 3. Parse subject and session identifiers using Regex
for file_path in all_nifti_files:
    full_path_str = str(file_path)
    filename = file_path.name
    
    # Extract Subject ID (matches OAS3 followed by digits, e.g., OAS30001)
    subj_match = re.search(r'(OAS3\d+)', full_path_str)
    
    # Extract Visit Day (matches d followed by digits, e.g., d0129 -> 129)
    day_match = re.search(r'[_\-]d(\d+)', full_path_str)
    
    # Extract full session label if available (e.g., OAS30001_MR_d0129)
    session_match = re.search(r'(OAS3\d+_[A-Za-z0-9]+_d\d+)', full_path_str)

    if subj_match:
        subject_id = subj_match.group(1)
        mri_day = int(day_match.group(1)) if day_match else None
        mr_session = session_match.group(1) if session_match else None

        data.append({
            "subject_id": subject_id,
            "MRI_Day": mri_day,
            "MR_Session": mr_session,
            "filename": filename,
            "file_path": str(file_path.resolve())
        })

# 4. Build DataFrame
df = pd.DataFrame(data)

print(f"Total files indexed: {len(df)}")
if not df.empty:
    print("\nSample Indexing Results:")
    print(df.head())

# 5. Save output CSV
csv_path = RESULTS_DIR / "oasis3_file_mapping.csv"
df.to_csv(csv_path, index=False)

print(f"\nDataFrame saved successfully to: {csv_path}")

Total NIfTI files found: 1791
Total files indexed: 1791

Sample Indexing Results:
  subject_id  MRI_Day         MR_Session  \
0   OAS30257      610  OAS30257_MR_d0610   
1   OAS30257      610  OAS30257_MR_d0610   
2   OAS30558     5591  OAS30558_MR_d5591   
3   OAS30383      134  OAS30383_MR_d0134   
4   OAS30383      134  OAS30383_MR_d0134   

                                   filename  \
0  sub-OAS30257_ses-d0610_run-01_T1w.nii.gz   
1  sub-OAS30257_ses-d0610_run-02_T1w.nii.gz   
2        sub-OAS30558_sess-d5591_T1w.nii.gz   
3  sub-OAS30383_ses-d0134_run-01_T1w.nii.gz   
4  sub-OAS30383_ses-d0134_run-02_T1w.nii.gz   

                                           file_path  
0  /home/ybrima/Data/OASIS3/OAS30257_MR_d0610/ana...  
1  /home/ybrima/Data/OASIS3/OAS30257_MR_d0610/ana...  
2  /home/ybrima/Data/OASIS3/OAS30558_MR_d5591/ana...  
3  /home/ybrima/Data/OASIS3/OAS30383_MR_d0134/ana...  
4  /home/ybrima/Data/OASIS3/OAS30383_MR_d0134/ana...  

DataFrame saved successfully to: result

In [19]:
import nibabel as nib


idx  = np.random.randint(df.shape[0])
# Pick the first image path
sample_row = df.iloc[idx]
sample_path = sample_row["file_path"]

print(f"Subject ID: {sample_row.get('Subject')}")
print(f"MR Session: {sample_row.get('MR_Session')}")
print(f"File Path:  {sample_path}\n")

# Load NIfTI header and data
img = nib.load(sample_path)
data = img.get_fdata()

print("--- Volume Properties ---")
print(f"Data Shape (Voxels): {data.shape}")
print(f"Data Type:           {data.dtype}")
print(f"Voxel Spacing (mm):  {img.header.get_zooms()}")

Subject ID: OAS30557
MR Session: OAS30557_MR_d2185
File Path:  /home/ybrima/Data/OASIS3/OAS30557_MR_d2185/anat2/sub-OAS30557_ses-d2185_run-01_T1w.nii.gz

--- Volume Properties ---
Data Shape (Voxels): (176, 256, 256)
Data Type:           float64
Voxel Spacing (mm):  (np.float32(0.9999997), np.float32(1.0), np.float32(1.0))


In [21]:

# RESULTS_DIR = Path("./results") # Adjust path if your files are in a different directory

def merge_oasis3_manifests(
    results_dir: Path = RESULTS_DIR,
    max_day_diff: int = 365,
    output_filename: str = "oasis3_unified_manifest.csv"
):
    mapping_csv = results_dir / "oasis3_file_mapping.csv"
    clinical_csv = results_dir / "oasis3_harmonized_clinical.csv"
    
    # Fallback to current directory if not found in results_dir
    if not mapping_csv.exists():
        mapping_csv = Path("oasis3_file_mapping.csv")
    if not clinical_csv.exists():
        clinical_csv = Path("oasis3_harmonized_clinical.csv")
        
    print(f"Loading file mapping from: {mapping_csv}")
    print(f"Loading clinical manifest from: {clinical_csv}")
    
    df_img = pd.read_csv(mapping_csv)
    df_clin = pd.read_csv(clinical_csv)

    # Standardize Subject key naming across both DataFrames
    subj_col_img = "Subject" if "Subject" in df_img.columns else "subject_id"
    subj_col_clin = "Subject" if "Subject" in df_clin.columns else "subject_id"
    
    df_img["Subject"] = df_img[subj_col_img].astype(str)
    df_clin["Subject"] = df_clin[subj_col_clin].astype(str)

    # Deduplicate image file paths up front
    df_img = df_img.drop_duplicates(subset=["file_path"]).reset_index(drop=True)
    
    merged_records = []
    
    print("\nMatching each unique image to its temporally closest clinical visit...")
    for idx, img_row in df_img.iterrows():
        subj = img_row["Subject"]
        mri_day = img_row.get("MRI_Day")
        file_path = img_row["file_path"]
        
        # Filter clinical evaluations for this subject
        subj_clin = df_clin[df_clin["Subject"] == subj].copy()
        if subj_clin.empty:
            continue
            
        if pd.notna(mri_day) and "Visit_Day" in subj_clin.columns:
            # Calculate absolute day difference between MRI scan and clinical visit
            subj_clin["day_delta"] = (subj_clin["Visit_Day"] - mri_day).abs()
            closest_idx = subj_clin["day_delta"].idxmin()
            closest_visit = subj_clin.loc[closest_idx]
            delta = closest_visit["day_delta"]
        else:
            # If days are not available, take the first available clinical record for the subject
            closest_visit = subj_clin.iloc[0]
            delta = np.nan

        # Enforce temporal window threshold
        if pd.isna(delta) or delta <= max_day_diff:
            # Merge image metadata with matched clinical record
            record = closest_visit.to_dict()
            record["file_path"] = file_path
            record["MR_Session"] = img_row.get("MR_Session", np.nan)
            record["MRI_Day"] = mri_day
            record["Day_Delta"] = delta
            merged_records.append(record)

    df_unified = pd.DataFrame(merged_records)

    # Explicitly drop any duplicate image file paths
    df_unified = df_unified.drop_duplicates(subset=["file_path"]).reset_index(drop=True)

    # Define standard column hierarchy
    id_cols = ["Subject", "MR_Session", "file_path", "MRI_Day", "Visit_Day", "Day_Delta", "Diagnosis"]
    predictor_cols = [c for c in df_unified.columns if c not in id_cols]
    
    df_unified = df_unified.reindex(columns=id_cols + predictor_cols)
    
    out_path = results_dir / output_filename
    results_dir.mkdir(parents=True, exist_ok=True)
    # df_unified.to_csv(out_path, index=False)

    print("\n--- Unified Merge Complete ---")
    print(f"Total Unique Image-Clinical Pairs Saved: {len(df_unified)}")
    print(f"Unique Subjects Represented: {df_unified['Subject'].nunique()}")
    print("\nClass Distribution:")
    print(df_unified["Diagnosis"].value_counts())
    
    if "Day_Delta" in df_unified.columns and df_unified["Day_Delta"].notna().any():
        print(f"\nMean Time Delta (MRI vs Clinical Visit): {df_unified['Day_Delta'].mean():.1f} days")
        print(f"Max Time Delta: {df_unified['Day_Delta'].max():.1f} days")
        
    print(f"\nSaved final unified manifest to: {out_path.resolve()}")
    return df_unified


df_unified = merge_oasis3_manifests()

Loading file mapping from: results/oasis3_file_mapping.csv
Loading clinical manifest from: results/oasis3_harmonized_clinical.csv

Matching each unique image to its temporally closest clinical visit...

--- Unified Merge Complete ---
Total Unique Image-Clinical Pairs Saved: 1703
Unique Subjects Represented: 533

Class Distribution:
Series([], Name: count, dtype: int64)

Mean Time Delta (MRI vs Clinical Visit): 99.9 days
Max Time Delta: 362.0 days

Saved final unified manifest to: /home/ybrima/dev/Scripts/results/oasis3_unified_manifest.csv


In [24]:
df_unified['Group'].value_counts()

Group
CN     1323
MCI     295
AD       85
Name: count, dtype: int64